# Evaluate the data agent in batch (preview)

Runs the Lab 2 test set against your **published** data agent and scores each answer with the Fabric data agent SDK.
Source: [Evaluate your data agent](https://learn.microsoft.com/fabric/data-science/evaluate-data-agent).

**Before you run it**

1. Publish the agent (Lab 2, step A7).
2. Create a lakehouse you can write to, for example `P07_Eval_LH` in your folder, and attach it to this notebook as the default lakehouse. The SDK writes two result tables to it.
3. Set your names in the next cell. Then select **Run all**.

In [ ]:
%pip install -U fabric-data-agent-sdk

In [ ]:
DATA_AGENT_NAME = "P07_Sales_Agent"
WORKSPACE_NAME = "ContosoPharma_Workshop"
TABLE_NAME = "p07_agent_evaluation"   # results: TABLE_NAME and TABLE_NAME + "_steps"
DATA_AGENT_STAGE = "production"       # "production" = published version, "sandbox" = draft

## Test set

The same questions and expected answers as `lab-02-data-agent/testing/ground_truth.csv`, generated from the data by `scripts/build_answer_key.py`.
Add the questions your business users gave you in Part B, step B4.

In [ ]:
import pandas as pd

df = pd.DataFrame(
    columns=["question", "expected_answer"],
    data=[
    ("What were total net sales in 2025?",
     "EUR 293.8 M"),
    ("Which therapeutic area had the highest net sales in 2025?",
     "Oncology (EUR 103.4 M)"),
    ("How much did Immunology net sales grow in 2025 compared with 2024?",
     "49.2% (EUR 43.4 M to EUR 64.7 M)"),
    ("What are the top 3 countries by net sales in 2026?",
     "Germany (EUR 37.9 M); France (EUR 30.1 M); United Kingdom (EUR 27.2 M)"),
    ("What was the discount % by channel in 2025?",
     "Direct 5.8%, Wholesale 12.0%, Tender 20.0%"),
    ("How many quality complaints were received for Zentora 100 mg tablets in Germany in April 2026?",
     "43"),
    ("In which weeks was Zentora 100 mg tablets out of stock in Germany?",
     "7 weeks: week ending 2026-05-03 to week ending 2026-06-14"),
    ("Which batch had the most quality complaints in 2026?",
     "ZT100-2603B (46 complaints)"),
    ("Why did net sales of Zentora 100 mg tablets in Germany drop in May 2026?",
     "Net sales fell from EUR 660 k (April) to EUR 167 k (May). Germany was out of stock from week ending 2026-05-03 to 2026-06-14, after 43 packaging complaints in April 2026 on batch ZT100-2603B (Blister foil torn on receipt; Blister seal not intact; Tablets loose in carton, blister seal open)."),
    ],
)
df

## Scoring rule

A custom critic prompt must contain `{query}`, `{expected_answer}` and `{actual_answer}`. This prompt normalizes units (k, M) before comparing, accepts rounding differences, and rejects a wrong order of magnitude.


In [ ]:
critic_prompt = """
Given the query, the expected answer and the actual answer, decide whether the actual answer is equivalent to the expected answer.
Before you compare amounts, convert both to the same unit: k = thousand, M = million, so "EUR 293.8 M" equals "EUR 293,800,000".
Amounts are equal only if they match after that conversion and after rounding to the precision used in the expected answer.
A different order of magnitude (for example k instead of M) is NOT equivalent.
Ignore currency symbols and thousands separators.
If the actual answer contains the expected facts plus extra correct context, it is equivalent.
If they are equivalent, respond with 'yes'.

Query: {query}

Expected Answer:
{expected_answer}

Actual Answer:
{actual_answer}

Is the actual answer equivalent to the expected answer?
"""

In [ ]:
from fabric.dataagent.evaluation import evaluate_data_agent

evaluation_id = evaluate_data_agent(
    df,
    DATA_AGENT_NAME,
    workspace_name=WORKSPACE_NAME,
    table_name=TABLE_NAME,
    data_agent_stage=DATA_AGENT_STAGE,
    critic_prompt=critic_prompt,
)
print(f"Evaluation ID: {evaluation_id}")

In [ ]:
from fabric.dataagent.evaluation import get_evaluation_summary

get_evaluation_summary(TABLE_NAME)

In [ ]:
from fabric.dataagent.evaluation import get_evaluation_details

# get_all_rows=False returns only the questions that weren't answered correctly
get_evaluation_details(evaluation_id, TABLE_NAME, get_all_rows=False, verbose=True)

## Next steps

- Fix the configuration in the draft (instructions, example queries, or Prep for AI in the model), publish again, and rerun this notebook. Each run is stored with its own evaluation ID.
- To check example queries before you add them, use `evaluate_few_shots` ([Example queries](https://learn.microsoft.com/fabric/data-science/data-agent-example-queries)).